In [ ]:
import requests
import pandas as pd
import matplotlib.pyplot as plt 
import frost_met_code_fred as fmc

# Insert your own client ID here
client_id = fmc.client_credential

#fmc_f.client_credentials
endpoint_prec = 'https://frost.met.no/observations/v0.jsonld'
parameters_prec = {
    'sources': 'SN17850',
    'elements': 'sum(precipitation_amount P1D)',
    'referencetime': '2025-01-01/2025-12-31',
}

# Issue an HTTP GET request
r_prec = requests.get(endpoint_prec, parameters_prec, auth=(client_id,''))
json_prec = r_prec.json()
data_precip = json_prec["data"]  # Endret til data_precip her

rows_precip = []

for i in range(len(data_precip)):
    row = pd.DataFrame(data_precip[i]['observations'])
    row['referenceTime'] = data_precip[i]['referenceTime']
    row['sourceId'] = data_precip[i]['sourceId']
    rows_precip.append(row)

df_precip = pd.concat(rows_precip, ignore_index=True)

# Converting time
df_precip["date"] = pd.to_datetime(df_precip["referenceTime"])
df_precip.head()

# makin 2 y-axis
fig, plt_1 = plt.subplots(figsize=(12, 5))

# Temperature
plt_1.plot(df["date"], df["value"], color="tab:red", label="temperature (2025)")
plt_1.set_xlabel("Date")
plt_1.set_ylabel("Temperature (°C)", color="tab:red")
plt_1.set_xticks(months)
plt_1.set_xticklabels(month_names)
plt_1.grid(True, linestyle="--", alpha=0.6)

# Precipitation
plt_2 = plt_1.twinx()
plt_2.bar(df_precip["date"], df_precip["value"], color="tab:blue", alpha=0.4, label="precipitation (2025)")
plt_2.set_ylabel("Precipitation (mm)", color="tab:blue")

plt.title("temperature and precipitation in Ås 2025")
plt.tight_layout()
plt.show()


# Summary table
summary_precip = pd.DataFrame({
    "Metrikk": ["Mean", "Median", "Minimum", "Maximum"],
    "Precipitation (mm)": [
        df_precip["value"].mean(),
        df_precip["value"].median(),
        df_precip["value"].min(),
        df_precip["value"].max()
    ]
})

print("Summary table for precipitation (2025)")
print(summary_precip.to_string(index=False))

In [ ]:
import requests
import pandas as pd
import frost_met_code_fred as fmc
import matplotlib.pyplot as plt
import yaml


# Insert your own client ID here
client_id = fmc.client_credentials


def number_of_heatwaves(station_id, year):
    endpoint = 'https://frost.met.no/observations/v0.jsonld'
    parameters = {
        'sources' : station_id,
        'elements' : 'max(air_temperature P1D)',
        'referencetime': f'{year}-01-01/{year}-12-31',
    }


# Issue an HTTP GET request
    r = requests.get(endpoint, parameters, auth=(client_id,''))

# Extract JSON data
    json = r.json()
    data = json['data']


    rows = []

    for i in range(len(data)):
        row = pd.DataFrame(data[i]['observations'])
        row['referenceTime'] = data[i]['referenceTime']
        row['sourceId'] = data[i]['sourceId']
        rows.append(row)



    df = pd.concat(rows, ignore_index=True)
    df.head()

    df["referenceTime"] = pd.to_datetime(df["referenceTime"])


    heatwaves = []
    current_streak = []

    for index, row in df.iterrows():
        if row["value"] >= 27:
            date_str = row["referenceTime"].strftime('%Y-%m-%d')
            current_streak.append(date_str)
        else:
            if len(current_streak) >= 5:
                heatwaves.append((current_streak[0], current_streak[-1]))
            current_streak = []


    yml_data= {
        'dates' : [f'{start} {end}' for start, end in heatwaves],
        'station_id' : station_id,
        'station_name' : 'Ås',
        'year' : year,
        'number_of_heatwaves' : len(heatwaves)
    }

    with open('heatwaves.yaml', 'w', encoding='utf-8') as file:
        yaml.dump(yml_data, file, sort_keys=False)

    print (f'it has been {len(heatwaves)} heatwaves in Ås')


number_of_heatwaves('SN17850', 2025)